# Integrated Gradients

In [4]:
# from transformers import AutoModelForSequenceClassification, AutoTokenizer
# from transformers_interpret import SequenceClassificationExplainer
# import pandas as pd
# from sklearn.preprocessing import LabelEncoder

# model_name = "C:/Users/anouk.xs4all/Projects/XS4ALL/saved_models/v2_3_entrtrian_loss_skf_fold4"
# model = AutoModelForSequenceClassification.from_pretrained(model_name)
# tokenizer = AutoTokenizer.from_pretrained("roberta-base")

# df = pd.read_json(r"C:\Users\anouk.xs4all\Projects\XS4ALL\training\train_w13.jsonl", lines=True)

# label_encoder = LabelEncoder()
# df['label'] = label_encoder.fit_transform(df['period_bucket'])

Loading weights: 100%|█████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 1639.52it/s]


![](img/confusion_matrix.v2.3_optimal_weights.png)

In [2]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers_interpret import SequenceClassificationExplainer
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import torch

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_v2_3_name = "C:/Users/anouk.xs4all/Projects/XS4ALL/saved_models/v2_3_optimal_weights_fold1"
model_v2_3 = AutoModelForSequenceClassification.from_pretrained(model_v2_3_name)
model_v2_3.eval()

tokenizer_v2_3 = AutoTokenizer.from_pretrained("roberta-base")

# Make sure to use the correct dataset for each model
df_13 = pd.read_json(r"C:\Users\anouk.xs4all\Projects\XS4ALL\training\train_w13.jsonl", lines=True)

label_encoder = LabelEncoder()
df_13['label'] = label_encoder.fit_transform(df_13['period_bucket'])

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [4]:
# After a few test runs some generators still showed,  but that is probably due to varying notations of e.g. 'FrontPage.Editor'

import sys
sys.path.append("C:/Users/anouk.xs4all/Projects/XS4ALL/script")
from labeling_function_v8 import SignalExtractor

# Hard-coding the masking, already defined in the model algorithm but just to be sure.
def mask_payload(payload):
    if not isinstance(payload, str):
        return ""

    for signal_name, pattern in SignalExtractor.regex_patterns.items():
        payload = pattern.sub("", payload)
    return payload

In [113]:
# Testing the masking
subset = df_13[df_13['label'] == 0]
subset_reset = subset.reset_index(drop=True)
samples = []
for idx, row in subset[10:15].iterrows():
    subset_reset['payload'] = mask_payload(row['payload'])
    samples.append({
        # 'url': subset_reset.iloc[idx].get('url', 'unknown'),
        # 'payload_preview': subset_reset.iloc[idx]['payload'][:500],
        'idx': idx
    })

print(samples)

[{'idx': 23}, {'idx': 27}, {'idx': 31}, {'idx': 34}, {'idx': 40}]


### 3.1 Targeting the misclassifications cases

In [5]:
from tqdm import tqdm

def missclassifications(model, tokenizer, df, true_label_idx, miss_label_idx, max_search=200):
    id2label = {0: "19971999", 1: "20002002", 2: "20032006", 3: "20072010"}
    
    true_str = id2label.get(true_label_idx)
    miss_str = id2label.get(miss_label_idx)

    # print("Searching for misclassifications...")

    subset = df[df['label'] == true_label_idx]

    count = 0

    subset_reset = subset.reset_index(drop=True)
    samples = []
    for idx, row in tqdm(subset_reset.iterrows(), total=len(subset_reset), desc="Calculating wrong predictions"):
        row['payload'] = mask_payload(row['payload'])
        
        if idx > max_search:
            print("Reached limit without finding more")
            break
            
        inputs = tokenizer(row['payload'], return_tensors="pt", truncation=True, max_length=512).to(model.device)

        # print(f"Test: {row['payload'][:200]}")
        
        with torch.no_grad():
            outputs = model(**inputs)
            
        predicted_idx = outputs.logits.argmax().item()

        # Check if this specific instance hits the error group
        if predicted_idx == miss_label_idx:
            samples.append({
                'url': subset_reset.iloc[idx].get('url', 'unknown'),
                # 'idx': idx,
                'payload': row['payload'],
                'true_label' : true_str,
                'pred_label' : id2label.get(predicted_idx)
            })

            if len(samples) >=3:
                break
    
    return samples

In [148]:
# We will load them again in the case studies below
# conf_97_03 = missclassifications(model_v2_3, tokenizer_v2_3, df_13, 0, 2)
# conf_97_07 = missclassifications(model_v2_3, tokenizer_v2_3, df_13, 0, 3)
# conf_03_97 = missclassifications(model_v2_3, tokenizer_v2_3, df_13, 2, 0)

In [149]:
def test(subset):
    item = subset[0]
    warc_payload = item['payload'] 
    url = item['url']
    return (warc_payload, 
            url, 
            print("--- Top Positive Tokens ---"), 
            print("=== Top Positive Tokens ==="))

print(test(conf_97_03))

--- Top Positive Tokens ---
=== Top Positive Tokens ===
('\n<!-- last modified on 07-01-2000 -->\n<html>\n<head>\n<meta name="Generator" content="UDO6 PL14 for TOS">\n<meta name="Keywords" content="Atari, ST, TT, Falcon, Hades, Milan, MagicMac">\n<meta name="Description" content="Stichting ST, Atari, Typefaces for the Atari">\n<meta name="Email" content="dziewon@xs4all.nl">\n<link rev=made href="mailto:dziewon@xs4all.nl" title="E-Mail">\n<link rel=home href="index.htm" title="Homepage">\n<title>Stichting ST - de Thuishaven van Atari in Nederland</title>\n</head>\n<frameset cols="30%,*" border=0 frameborder=0 framespacing=0>\n\t<frame src="00toc.htm" name="UDOtoc" marginwidth=0 marginheight=0>\n\t<frame src="00con.htm" name="UDOcon" marginwidth=0 marginheight=0>\n</frameset>\n</html>\n\n\n\n<!-- last modified on 07-01-2000 -->\n<html>\n<head>\n<title>Inhoud</title>\n<meta name="Generator" content="UDO6 PL14 for TOS">\n<meta name="Email" content="dziewon@xs4all.nl">\n<link re', 'www.atar

### Denoising token fragmentation

In [6]:
import re
import string
import numpy as np

# ==========================================
# 1. NOISE FILTERING
# ==========================================
noise_tokens = {
    '<s>', '</s>', '<pad>', '<unk>', '<mask>',
    *list(string.printable)
}

def is_noise_token(token):
    # Remove special RoBERTa prefix
    clean = token.replace('Ġ', '').replace('Ċ', '').strip()
    
    if not clean:
        return True
    if clean in noise_tokens:
        return True
    if re.match(r'^\d+$', clean):  # filter isolated numbers
        return True
    if len(clean) <= 1:
        return True

    return False


# ==========================================
# 2. INDEX-AWARE BPE MERGING
# ==========================================
def merge_bpe_attributions_with_indices(tokens, attributions):
    """
    Merges sub-tokens and averages their attributions, while explicitly tracking 
    the original list indices of every sub-token for precise downstream masking.
    """
    merged_tokens, merged_attrs, merged_indices = [], [], []
    current_token, current_attrs, current_idx_group = "", [], []
    
    for i, (token, attr) in enumerate(zip(tokens, attributions)):
        clean_token = token.replace('Ġ', ' ').replace('Ċ', '')
        
        if token.startswith('Ġ') or token.startswith('<'):
            if current_token:
                merged_tokens.append(current_token)
                merged_attrs.append(np.mean(current_attrs))
                merged_indices.append(current_idx_group)
            
            # Start a new token buffer
            current_token = clean_token.lstrip() 
            current_attrs = [attr]
            current_idx_group = [i]
        else:
            # Append to existing token buffer
            current_token += clean_token
            current_attrs.append(attr)
            current_idx_group.append(i)
            
    # Flush the final buffer
    if current_token:
        merged_tokens.append(current_token)
        merged_attrs.append(np.mean(current_attrs))
        merged_indices.append(current_idx_group)
        
    return merged_tokens, merged_attrs, merged_indices

### 3.2 Visualizations and manual inspections

In [7]:
# ==========================================
# 3. MAIN VISUALIZATION PIPELINE
# ==========================================
def visualization(model, tokenizer, subset, item_index=0):
    cls_explainer = SequenceClassificationExplainer(model, tokenizer)
    model.config.id2label = {0: "1997-1999", 1: "2000-2002", 2: "2003-2006", 3: "2007-2010"}

    item = subset[item_index]
    warc_payload = item['payload']

    # ===== ORIGINAL PAYLOAD EXPLANATION =====
    attributions = cls_explainer(warc_payload)

    tokens = [a[0] for a in attributions]
    scores = [a[1] for a in attributions]

    # Merge tokens and retrieve their absolute indices
    merged_tokens, merged_scores, merged_indices = merge_bpe_attributions_with_indices(tokens, scores)

    # Build dataframe with the Index tracker column
    attr_df = pd.DataFrame({
        'Token': merged_tokens, 
        'Score': merged_scores,
        'Indices': merged_indices
    })
    
    # Filter noise and reset index
    attr_df = attr_df[~attr_df['Token'].apply(is_noise_token)].reset_index(drop=True)

    # Extract the top 10 positive tokens and their target indices
    top_10_df = attr_df.sort_values(by='Score', ascending=False).head(10)
    top_tokens = top_10_df['Token'].tolist()
    
    # Flatten the list of lists into a single list of target indices
    indices_to_mask = []
    for idx_group in top_10_df['Indices']:
        indices_to_mask.extend(idx_group)

    print(f"Top 10 attributed tokens: {top_tokens}")

    # ===== MASKED PAYLOAD GENERATION (INDEX PROTOCOL) =====
    # 1. Tokenize the original payload to get the exact tensor array
    inputs = tokenizer(warc_payload, truncation=True, max_length=512)
    input_ids = inputs['input_ids']

    # 2. Iterate through our target indices and overwrite them with the model's standard <mask> ID
    for idx in indices_to_mask:
        # Failsafe check to ensure index is within bounds of truncated sequence
        if idx < len(input_ids): 
            input_ids[idx] = tokenizer.mask_token_id

    # 3. Decode the exact tensor array back into a text string for the explainer to read.
    # We strip the <s> and </s> tokens ([1:-1]) so the explainer doesn't double-wrap the sequence.
    masked_payload = tokenizer.decode(input_ids[1:-1])


    # ===== RUN EXPLAINER ON MASKED PAYLOAD =====
    attributions_masked = cls_explainer(masked_payload)
    
    tokens_masked = [a[0] for a in attributions_masked]
    scores_masked = [a[1] for a in attributions_masked]

    # We don't need indices for the masked visualization DataFrame
    merged_tokens_masked, merged_scores_masked, _ = merge_bpe_attributions_with_indices(tokens_masked, scores_masked)
    
    attr_df_masked = pd.DataFrame({'Token': merged_tokens_masked, 'Score': merged_scores_masked})
    attr_df_masked = attr_df_masked[~attr_df_masked['Token'].apply(is_noise_token)].reset_index(drop=True)

    # ===== PRINT RESULTS =====
    print(f"\nURL: {item['url']}")
    print(f"True label: {item['true_label']}")

    print("\n=== ORIGINAL PAYLOAD ===")
    print("Top Positive Tokens:")
    print(attr_df[['Token', 'Score']].sort_values(by='Score', ascending=False).head(10))
    print("\nTop Negative Tokens:")
    print(attr_df[['Token', 'Score']].sort_values(by='Score', ascending=True).head(10))

    print("\n=== MASKED PAYLOAD ===")
    print("Top Positive Tokens:")
    print(attr_df_masked.sort_values(by='Score', ascending=False).head(10))
    print("\nTop Negative Tokens:")
    print(attr_df_masked.sort_values(by='Score', ascending=True).head(10))

    # ===== HTML VISUALIZATIONS =====
    # Restore state on original payload
    cls_explainer(warc_payload)
    viz_original = cls_explainer.visualize(true_class=item['true_label'])

    # Update state on masked payload
    cls_explainer(masked_payload)
    viz_masked = cls_explainer.visualize(true_class=item['true_label'])

    return viz_original, viz_masked, top_tokens, attr_df, attr_df_masked

#### 3.2.1 1997–1999 $\leftrightarrow$ 2003–2006
Why does a 90s website look like the late 2000s to RoBERTa?

In [9]:
# Load the sample subset (n=3)
conf_97_03 = missclassifications(model_v2_3, tokenizer_v2_3, df_13, 0, 2)

Calculating wrong predictions:   6%|██▉                                               | 27/455 [00:22<05:53,  1.21it/s]


In [45]:
# Visualize the attribution scores of the first sample (change item_index) 
visualization(model_v2_3, tokenizer_v2_3, conf_97_03, item_index=0)

Top 10 attributed tokens: ['<!--lastmodifiedon07-01-2000-->', '<metaname="Generator"content="UDO6PL14forTOS">', '<head>', '<metaname="Generator"content="UDO6PL14forTOS">', '</title>', '<html>', '<html>', '<metaname="Description"content="StichtingST,Atari,TypefacesfortheAtari">', '</head>', '<metaname="Keywords"content="Atari,ST,TT,Falcon,Hades,Milan,MagicMac">']

URL: www.atarist.home.xs4all.nl
True label: 19971999

=== ORIGINAL PAYLOAD ===
Top Positive Tokens:
                                                Token     Score
0                     <!--lastmodifiedon07-01-2000-->  0.089493
3      <metaname="Generator"content="UDO6PL14forTOS">  0.059825
19                                             <head>  0.054798
22     <metaname="Generator"content="UDO6PL14forTOS">  0.028110
10                                           </title>  0.015160
18                                             <html>  0.014341
1                                              <html>  0.010682
5   <metaname="Descrip

(<IPython.core.display.HTML object>,
 <IPython.core.display.HTML object>,
 ['<!--lastmodifiedon07-01-2000-->',
  '<metaname="Generator"content="UDO6PL14forTOS">',
  '<head>',
  '<metaname="Generator"content="UDO6PL14forTOS">',
  '</title>',
  '<html>',
  '<html>',
  '<metaname="Description"content="StichtingST,Atari,TypefacesfortheAtari">',
  '</head>',
  '<metaname="Keywords"content="Atari,ST,TT,Falcon,Hades,Milan,MagicMac">'],
                                                 Token     Score  \
 0                     <!--lastmodifiedon07-01-2000-->  0.089493   
 1                                              <html>  0.010682   
 2                                              <head>  0.004659   
 3      <metaname="Generator"content="UDO6PL14forTOS">  0.059825   
 4   <metaname="Keywords"content="Atari,ST,TT,Falco...  0.007674   
 5   <metaname="Description"content="StichtingST,At...  0.009073   
 6       <metaname="Email"content="dziewon@xs4all.nl"> -0.000974   
 7   <linkrev=madehref=

In [10]:
visualization(model_v2_3, tokenizer_v2_3, conf_97_03, item_index=1)

Top 10 attributed tokens: ['<metaname="ProgId"content="FrontPage.Editor.Document">', '<a.length;i++)if(a[i].indexOf("#")!=0){d.MM_p[j]=newImage;d.MM_p[j++].src=a[i];}}}functionMM_findObj(n,d){//v4.01varp,i,x;if(!d)d=document;if((p=n.indexOf("?"))>0&&parent.frames.length){d=parent.frames[n.substring(p+1)].document;n=n.substring(0,p);}if(!(x=d[n])&&d.all)x=d.all[n];for(i=0;!x&&i', '<d.forms.length;i++)x=d.forms[i][n];for(i=0;!x&&d.layers&&i', '<title>AccountantskantoorVerheggen', '<d.layers.length;i++)x=MM_findObj(n,d.layers[i].document);if(!x&&d.getEl', '<metahttp-equiv="Content-Type"content="text/html;charset=windows-1252">">', '<!--functionMM_preloadImages(){//v3.0vard=document;if(d.images){if(!d.MM_p)d.MM_p=newArray();vari,j=d.MM_p.length,a=MM_preloadImages.arguments;for(i=0;i', '<scriptlanguage="JavaScript"type="text/JavaScript">', '<head>', '<metaname="MicrosoftBorder"content="tl,default">']

URL: www.verhegge.home.xs4all.nl
True label: 19971999

=== ORIGINAL PAYLOAD ===
Top Positi

(<IPython.core.display.HTML object>,
 <IPython.core.display.HTML object>,
 ['<metaname="ProgId"content="FrontPage.Editor.Document">',
  '<a.length;i++)if(a[i].indexOf("#")!=0){d.MM_p[j]=newImage;d.MM_p[j++].src=a[i];}}}functionMM_findObj(n,d){//v4.01varp,i,x;if(!d)d=document;if((p=n.indexOf("?"))>0&&parent.frames.length){d=parent.frames[n.substring(p+1)].document;n=n.substring(0,p);}if(!(x=d[n])&&d.all)x=d.all[n];for(i=0;!x&&i',
  '<d.forms.length;i++)x=d.forms[i][n];for(i=0;!x&&d.layers&&i',
  '<title>AccountantskantoorVerheggen',
  '<d.layers.length;i++)x=MM_findObj(n,d.layers[i].document);if(!x&&d.getEl',
  '<metahttp-equiv="Content-Type"content="text/html;charset=windows-1252">">',
  '<!--functionMM_preloadImages(){//v3.0vard=document;if(d.images){if(!d.MM_p)d.MM_p=newArray();vari,j=d.MM_p.length,a=MM_preloadImages.arguments;for(i=0;i',
  '<scriptlanguage="JavaScript"type="text/JavaScript">',
  '<head>',
  '<metaname="MicrosoftBorder"content="tl,default">'],
                       

In [15]:
visualization(model_v2_3, tokenizer_v2_3, conf_97_03, item_index=2)

Top 10 attributed tokens: ['</TITLE>', '<CENTER>', '<HEAD>', '<TITLE>TheIcemansHomeSteadontheWWW', '<HTML>', '<BR>', '<METAname="description"content="Underconstruction">', '<BODYbgcolor="#000000"text="#00FF00"link="#66FFFF">', '</font></b></p>', '<METAname="keywords"content="Iceman,Nederland,">']

URL: www.iceman.home.xs4all.nl
True label: 19971999

=== ORIGINAL PAYLOAD ===
Top Positive Tokens:
                                                Token     Score
3                                            </TITLE>  0.094857
12                                           <CENTER>  0.051868
1                                              <HEAD>  0.044628
2                  <TITLE>TheIcemansHomeSteadontheWWW  0.035764
0                                              <HTML>  0.012320
11                                               <BR>  0.010001
6   <METAname="description"content="Underconstruct...  0.007911
10  <BODYbgcolor="#000000"text="#00FF00"link="#66F...  0.004058
14                        

(<IPython.core.display.HTML object>,
 <IPython.core.display.HTML object>,
 ['</TITLE>',
  '<CENTER>',
  '<HEAD>',
  '<TITLE>TheIcemansHomeSteadontheWWW',
  '<HTML>',
  '<BR>',
  '<METAname="description"content="Underconstruction">',
  '<BODYbgcolor="#000000"text="#00FF00"link="#66FFFF">',
  '</font></b></p>',
  '<METAname="keywords"content="Iceman,Nederland,">'],
                                                 Token     Score  \
 0                                              <HTML>  0.012320   
 1                                              <HEAD>  0.044628   
 2                  <TITLE>TheIcemansHomeSteadontheWWW  0.035764   
 3                                            </TITLE>  0.094857   
 4   <METAHTTP-EQUIV="Content-Type"CONTENT="text/ht... -0.034166   
 5    <METAname="keywords"content="Iceman,Nederland,">  0.001802   
 6   <METAname="description"content="Underconstruct...  0.007911   
 7   <METAname="author"content="Thel@rsh,http://www... -0.008739   
 8         <METAname="

#### 3.2.2. 1997–1999 $\rightarrow$ 2007–2010

In [11]:
# Load the sample subset (n=3)
conf_97_07 = missclassifications(model_v2_3, tokenizer_v2_3, df_13, 0, 3)

Calculating wrong predictions:   2%|▊                                                  | 7/455 [00:05<06:19,  1.18it/s]


In [12]:
visualization(model_v2_3, tokenizer_v2_3, conf_97_07, item_index=0)

Top 10 attributed tokens: ['<metaname="ProgId"content="FrontPage.Editor.Document">', '</title>">', '</head>', '</a', '<head>', '<framename="onder"src="start1.htm"target="_self">', '<ahref="mailto:Jeanmi@mobistar.be">Jean-MichelBuess', '<title>Reto2002', '<framesetrows="13%,*"framespacing="0"border="0"frameborder="0">', '<p>Opdezepaginawordenframesgebruikt,maaruwbrowserondersteuntgeenframes.']

URL: www.reto2002.home.xs4all.nl
True label: 19971999

=== ORIGINAL PAYLOAD ===
Top Positive Tokens:
                                                Token     Score
5   <metaname="ProgId"content="FrontPage.Editor.Do...  0.101491
4                                          </title>">  0.033572
7                                             </head>  0.025408
18                                                </a  0.012032
1                                              <head>  0.011267
10  <framename="onder"src="start1.htm"target="_self">  0.008992
17  <ahref="mailto:Jeanmi@mobistar.be">Jean-Michel... 

(<IPython.core.display.HTML object>,
 <IPython.core.display.HTML object>,
 ['<metaname="ProgId"content="FrontPage.Editor.Document">',
  '</title>">',
  '</head>',
  '</a',
  '<head>',
  '<framename="onder"src="start1.htm"target="_self">',
  '<ahref="mailto:Jeanmi@mobistar.be">Jean-MichelBuess',
  '<title>Reto2002',
  '<framesetrows="13%,*"framespacing="0"border="0"frameborder="0">',
  '<p>Opdezepaginawordenframesgebruikt,maaruwbrowserondersteuntgeenframes.'],
                                                 Token     Score  \
 0                                              <html> -0.006141   
 1                                              <head>  0.011267   
 2   <metahttp-equiv="Content-Type"content="text/ht... -0.012876   
 3                                     <title>Reto2002  0.003072   
 4                                          </title>">  0.033572   
 5   <metaname="ProgId"content="FrontPage.Editor.Do...  0.101491   
 6     <metaname="MicrosoftBorder"content="b,default"> -0.00

In [14]:
visualization(model_v2_3, tokenizer_v2_3, conf_97_07, item_index=1)

Top 10 attributed tokens: ['<metahttp-equiv="Content-Type"content="text/html;charset=windows-1252">">', '</html>', '<pstyle="line-height:120%"><fontface="ArialBla', '</p>', '</noframes>', '<title>Biografie', '<bodybgcolor="#000080">', '</title>">', '<palign="center"><fontface="ArialBlack"size="4"color="#FFFFFF">Inhoud&nbsp;&nbsp;', '<head>']

URL: www.wvegmond.home.xs4all.nl
True label: 19971999

=== ORIGINAL PAYLOAD ===
Top Positive Tokens:
                                                Token     Score
21  <metahttp-equiv="Content-Type"content="text/ht...  0.053199
17                                            </html>  0.027871
30     <pstyle="line-height:120%"><fontface="ArialBla  0.007044
13                                               </p>  0.005726
15                                        </noframes>  0.004412
23                                   <title>Biografie  0.003987
27                            <bodybgcolor="#000080">  0.002493
3                                         

(<IPython.core.display.HTML object>,
 <IPython.core.display.HTML object>,
 ['<metahttp-equiv="Content-Type"content="text/html;charset=windows-1252">">',
  '</html>',
  '<pstyle="line-height:120%"><fontface="ArialBla',
  '</p>',
  '</noframes>',
  '<title>Biografie',
  '<bodybgcolor="#000080">',
  '</title>">',
  '<palign="center"><fontface="ArialBlack"size="4"color="#FFFFFF">Inhoud&nbsp;&nbsp;',
  '<head>'],
                                                 Token     Score  \
 0                                              <html> -0.048323   
 1                                              <head> -0.008862   
 2                                     <title>NewPage1 -0.012933   
 3                                          </title>">  0.002098   
 4   <metaname="ProgId"content="FrontPage.Editor.Do... -0.007397   
 5                                 <basetarget="main"> -0.010438   
 6                                             </head> -0.004559   
 7                              <framesetcol

In [13]:
visualization(model_v2_3, tokenizer_v2_3, conf_97_07, item_index=2)

KeyboardInterrupt: 

#### 3.2.3. 2003–2006 $\rightarrow$ 1997–1999 (41 instances)
Why are mid-2000s pages leaking into the 90s bucket?

In [15]:
# Load the sample subset (n=3)
conf_03_97 = missclassifications(model_v2_3, tokenizer_v2_3, df_13, 2, 0)

Calculating wrong predictions:  11%|█████▋                                            | 14/123 [00:10<01:18,  1.38it/s]


In [16]:
visualization(model_v2_3, tokenizer_v2_3, conf_03_97, item_index=0)

Top 10 attributed tokens: ['ER', 'ER', 'IGN', '>', 'TIT', '>', 'BR', 'LE', 'IGHT', 'CENT']


KeyboardInterrupt: 

In [ ]:
visualization(model_v2_3, tokenizer_v2_3, conf_03_97, item_index=1)

In [ ]:
visualization(model_v2_3, tokenizer_v2_3, conf_03_97, item_index=2)

### 3.3 Correct classifications

#### 3.3.1 2007–2010 $\leftrightarrow$ 2007-2010
What constitutes a correct 2007-2010 prediction?

In [151]:
conf_07_07 = missclassifications(model_v2_3, tokenizer_v2_3, df_13, 3, 3)

Calculating wrong predictions:   3%|█▋                                                  | 3/92 [00:03<01:56,  1.31s/it]


In [152]:
visualization(model_v2_3, tokenizer_v2_3, conf_07_07, item_index=0)

=== Top Positive Tokens ===
       Token     Score
144      css  0.472106
69        Id  0.317359
67       Pro  0.259878
254        !  0.254467
66        ="  0.135876
68         g  0.129045
143        /  0.116009
255       --  0.099221
51         Ċ  0.077884
65      name  0.075599
74      Page  0.063483
73     Front  0.057967
146     href  0.053532
145        "  0.052465
258  igation  0.047595

=== Top Negative Tokens ===
     Token     Score
48      12 -0.468956
256     ms -0.306668
49      52 -0.165485
47       - -0.159384
50      "> -0.125668
43   chars -0.096795
315     MS -0.075283
72      =" -0.071627
123      > -0.069026
44      et -0.062829
316      F -0.052775
314      Ċ -0.045858
6        > -0.038435
70       " -0.035492
59       > -0.034783


(<IPython.core.display.HTML object>,
 'www.oene.home.xs4all.nl',
 None,
 None,
 None,
 None)

In [52]:
cls_explainer.visualize()

In [49]:
# 1. Generate the visualization object without saving internally
html_output = cls_explainer.visualize()

# 2. Manually write the data with explicit UTF-8 encoding
with open("test_transformers_interpret_237.html", "w", encoding="utf-8") as f:
    f.write(html_output.data)

print("File successfully saved with UTF-8 encoding.")

File successfully saved with UTF-8 encoding.
